# THUMOS14 Two-Stage TAL — v3 (with class labels)

**Stage 1 (BiMambaGlobalScanner)** — frame-level binary "is this an action?" scanner.
**Stage 2 (TransformerRefiner)** — per-proposal head that predicts:
  1. boundary residual `(Δstart, Δend)` for refinement
  2. **class label** (one of 20 THUMOS14 classes) for action recognition

## What's new vs v2

- 20-way action classification per proposal
- Per-class mAP evaluation matching the official THUMOS14 metric
- Multi-task loss for the refiner: GIoU + smooth-L1 + cross-entropy
- Removed the dataset-download cell (handled by `bootstrap.sh` now)

## Prerequisites

Run `bash /workspace/bootstrap.sh` first. That handles env, mamba-ssm, dataset, RAM stage.
Then in VS Code, select kernel `/workspace/tal_training/venv/bin/python`.


In [ ]:
import os, sys, json, time, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────
DATA_ROOT_PERSISTENT = '/workspace/THUMOS14'
DATA_ROOT_FAST       = '/dev/shm/THUMOS14'
DATA_ROOT = DATA_ROOT_FAST if os.path.exists(DATA_ROOT_FAST) else DATA_ROOT_PERSISTENT
assert os.path.exists(f'{DATA_ROOT}/gt.json'), \
    f'Dataset not found at {DATA_ROOT} — run bootstrap.sh first'

CACHE_DIR    = '/workspace/cache'
CACHE_TRAIN  = f'{CACHE_DIR}/cache_train.pt'
CACHE_TEST   = f'{CACHE_DIR}/cache_test.pt'

CKPT_DIR     = '/workspace/checkpoints'
SCANNER_CKPT = f'{CKPT_DIR}/mamba_scanner_best.pth'
REFINER_CKPT = f'{CKPT_DIR}/transformer_refiner_best.pth'
PRED_FILE    = '/workspace/submission_predictions.json'
CLASS_INDEX_FILE = f'{CACHE_DIR}/class_index.json'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)

# ── Data / model constants ─────────────────────────────────────────────
FPS, CLIP_LENGTH, WINDOW_SIZE = 25.0, 16, 64
INPUT_DIM = 2048

# ── Hyperparameters ────────────────────────────────────────────────────
BATCH_SIZE      = 4
MAX_SEQ_LEN     = 1024
EPOCHS_STAGE1   = 10
EPOCHS_STAGE2   = 20
LR              = 1e-4
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0
FOCAL_ALPHA     = 0.25
FOCAL_GAMMA     = 2.0
SCORE_THRESHOLD = 0.4
NMS_IOU_THR     = 0.4

# Multi-task loss weights for refiner
W_GIOU = 1.0
W_L1   = 0.5
W_CLS  = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB VRAM')
print(f'DATA_ROOT = {DATA_ROOT}')


In [ ]:
# Build a stable string-label -> integer-index mapping from the GT.
# This handles both 'label' (string class name) and 'label_id' (integer) keys
# that THUMOS14 distributions sometimes use.

def _label_of(ann):
    if 'label' in ann:    return ann['label']
    if 'label_id' in ann: return f"class_{ann['label_id']}"
    raise KeyError(f'No label found in annotation: {ann}')


if os.path.exists(CLASS_INDEX_FILE):
    with open(CLASS_INDEX_FILE) as f:
        idx_data = json.load(f)
    CLASS_NAMES  = idx_data['classes']
    CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
    print(f'Loaded class index from cache: {len(CLASS_NAMES)} classes')
else:
    with open(f'{DATA_ROOT}/gt.json') as f:
        gt = json.load(f)
    seen = set()
    for vid, entry in gt.get('database', {}).items():
        for ann in entry.get('annotations', []):
            seen.add(_label_of(ann))
    CLASS_NAMES  = sorted(seen)
    CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
    with open(CLASS_INDEX_FILE, 'w') as f:
        json.dump({'classes': CLASS_NAMES}, f, indent=2)
    print(f'Built class index: {len(CLASS_NAMES)} classes -> {CLASS_INDEX_FILE}')

NUM_CLASSES = len(CLASS_NAMES)
print(f'NUM_CLASSES = {NUM_CLASSES}')
print(f'Sample: {CLASS_NAMES[:5]}{"..." if NUM_CLASSES > 5 else ""}')


In [ ]:
# Each cached video is now (fused, mask, vid, segments) where `segments` is a
# list of (start_idx, end_idx, class_idx) triples. Stage-1 dataset uses
# (fused, mask). Refiner dataset uses `segments`.

def build_cache(split, out_path):
    if os.path.exists(out_path):
        # Quick check: does it contain the new 4-tuple format?
        sample = torch.load(out_path, weights_only=False)[0]
        if len(sample) == 4:
            print(f'[{split}] cache exists at {out_path} (new format) — skipping rebuild')
            return
        print(f'[{split}] cache exists but is old format — rebuilding')

    rgb_dir, flow_dir = f'{DATA_ROOT}/features/{split}/rgb', f'{DATA_ROOT}/features/{split}/flow'
    with open(f'{DATA_ROOT}/split_{split}.txt') as f:
        vids = [l.strip() for l in f if l.strip()]
    with open(f'{DATA_ROOT}/gt.json') as f:
        gt = json.load(f)

    cache, skipped, t0 = [], 0, time.time()
    print(f'[{split}] caching {len(vids)} videos from {DATA_ROOT}...', flush=True)
    for i, vid in enumerate(vids):
        if i % 25 == 0:
            print(f'  {i}/{len(vids)}  cached={len(cache)}  skipped={skipped}', flush=True)
        rgb_p, flow_p = f'{rgb_dir}/{vid}.npy', f'{flow_dir}/{vid}.npy'
        if not (os.path.exists(rgb_p) and os.path.exists(flow_p)):
            skipped += 1; continue
        try:
            rgb, flow = np.load(rgb_p), np.load(flow_p)
        except Exception as e:
            print(f'  [WARN] {vid}: {e}', flush=True); skipped += 1; continue

        fused = torch.from_numpy(np.concatenate([rgb, flow], axis=-1)).float()
        T = fused.shape[0]
        mask = torch.zeros(T, dtype=torch.float32)
        segments = []   # list of (s_idx, e_idx, class_idx)

        for ann in gt.get('database', {}).get(vid, {}).get('annotations', []):
            label = _label_of(ann)
            if label not in CLASS_TO_IDX: continue          # skip unknown classes
            cls = CLASS_TO_IDX[label]
            s = max(0, min(int(float(ann['segment'][0]) * FPS / CLIP_LENGTH), T - 1))
            e = max(0, min(int(float(ann['segment'][1]) * FPS / CLIP_LENGTH), T - 1))
            if s <= e:
                mask[s:e+1] = 1.0
                segments.append((s, e, cls))

        cache.append((fused, mask, vid, segments))

    print(f'[{split}] done: cached={len(cache)} skipped={skipped} ({time.time()-t0:.1f}s)', flush=True)
    torch.save(cache, out_path)
    print(f'[{split}] saved to {out_path}', flush=True)


build_cache('train', CACHE_TRAIN)
build_cache('test',  CACHE_TEST)


In [ ]:
class CachedScannerDataset(Dataset):
    """Stage-1 dataset: returns (fused_features, frame_mask, vid_id).
    Class info is in the cache too but we ignore it here — Stage 1 is binary."""
    def __init__(self, cache_path, max_len=None):
        self.cache = torch.load(cache_path, weights_only=False)
        self.max_len = max_len
    def __len__(self): return len(self.cache)
    def __getitem__(self, idx):
        fused, mask, vid, _segments = self.cache[idx]
        if self.max_len is not None and fused.shape[0] > self.max_len:
            start = random.randint(0, fused.shape[0] - self.max_len)
            fused = fused[start:start + self.max_len]
            mask  = mask [start:start + self.max_len]
        return fused, mask, vid


def pad_collate(batch):
    feats, masks, ids = zip(*batch)
    lengths   = torch.tensor([f.shape[0] for f in feats], dtype=torch.long)
    feats_p   = pad_sequence(feats, batch_first=True)
    masks_p   = pad_sequence(masks, batch_first=True, padding_value=0.0)
    pad_mask  = torch.arange(feats_p.size(1))[None, :] >= lengths[:, None]
    return feats_p, masks_p, pad_mask, list(ids)


train_dataset = CachedScannerDataset(CACHE_TRAIN, max_len=MAX_SEQ_LEN)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=False, collate_fn=pad_collate)

feats, mask, pad_mask, ids = next(iter(train_loader))
print(f'feats={tuple(feats.shape)}  mask_pos={int(mask.sum())}/{mask.numel()}  '
      f'valid={int((~pad_mask).sum())}/{pad_mask.numel()}')


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha, self.gamma, self.reduction = alpha, gamma, reduction
    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p    = torch.sigmoid(logits)
        p_t  = p * targets + (1 - p) * (1 - targets)
        a_t  = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = a_t * (1 - p_t).pow(self.gamma) * bce
        if self.reduction == 'mean': return loss.mean()
        if self.reduction == 'sum':  return loss.sum()
        return loss


try:
    from mamba_ssm import Mamba as MambaSSMBlock
    HAS_MAMBA_SSM = True
    print('[Mamba] using mamba_ssm (CUDA selective scan)')
except Exception as e:
    HAS_MAMBA_SSM = False
    from transformers import MambaConfig, MambaModel
    print(f'[Mamba] mamba_ssm unavailable ({e}); using transformers fallback')


class BiMambaGlobalScanner(nn.Module):
    def __init__(self, input_dim=2048, d_model=256, n_layers=4,
                 d_state=16, d_conv=4, expand=2, kernel_size=7):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.temporal_smooth = nn.Conv1d(d_model, d_model, kernel_size=kernel_size,
                                         padding=kernel_size // 2, groups=d_model)
        if HAS_MAMBA_SSM:
            self.fwd_blocks = nn.ModuleList([
                MambaSSMBlock(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
                for _ in range(n_layers)])
            self.bwd_blocks = nn.ModuleList([
                MambaSSMBlock(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
                for _ in range(n_layers)])
            self.fwd_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
            self.bwd_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        else:
            cfg = MambaConfig(hidden_size=d_model, state_size=d_state, conv_kernel=d_conv,
                              expand=expand, num_hidden_layers=n_layers)
            self.fwd_hf = MambaModel(cfg)
            self.bwd_hf = MambaModel(cfg)
        self.classifier = nn.Linear(d_model * 2, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.temporal_smooth(x.transpose(1, 2)).transpose(1, 2)
        if HAS_MAMBA_SSM:
            f = x
            for blk, ln in zip(self.fwd_blocks, self.fwd_norms):
                f = f + blk(ln(f))
            b = torch.flip(x, dims=[1])
            for blk, ln in zip(self.bwd_blocks, self.bwd_norms):
                b = b + blk(ln(b))
            b = torch.flip(b, dims=[1])
        else:
            f = self.fwd_hf(inputs_embeds=x).last_hidden_state
            b = torch.flip(self.bwd_hf(inputs_embeds=torch.flip(x, dims=[1])).last_hidden_state, dims=[1])
        h = torch.cat([f, b], dim=-1)
        return self.classifier(h).squeeze(-1)


_m = BiMambaGlobalScanner().to(DEVICE)
print(f'Stage1 params: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M')
del _m


In [ ]:
# Skip this cell if you already have mamba_scanner_best.pth from a previous run.
# Stage 1 architecture has not changed since v2 — checkpoints are compatible.

mamba_model = BiMambaGlobalScanner().to(DEVICE)
focal_none  = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, reduction='none').to(DEVICE)
optimizer   = optim.AdamW(mamba_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler      = torch.amp.GradScaler('cuda')

best_loss = float('inf')

for epoch in range(EPOCHS_STAGE1):
    mamba_model.train()
    total, n = 0.0, 0
    pbar = tqdm(train_loader, desc=f'Stage1 {epoch+1}/{EPOCHS_STAGE1}',
                mininterval=1.0, maxinterval=10.0)
    for feats, target_mask, pad_mask, _ in pbar:
        feats       = feats.to(DEVICE, non_blocking=True)
        target_mask = target_mask.to(DEVICE, non_blocking=True)
        pad_mask    = pad_mask.to(DEVICE, non_blocking=True)
        valid       = (~pad_mask).float()

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits    = mamba_model(feats)
            per_frame = focal_none(logits.float(), target_mask)
            loss      = (per_frame * valid).sum() / valid.sum().clamp(min=1.0)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(mamba_model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()

        total += loss.item(); n += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg = total / max(1, n)
    print(f'epoch {epoch+1}: avg_loss={avg:.4f}')
    if avg < best_loss:
        best_loss = avg
        torch.save(mamba_model.state_dict(), SCANNER_CKPT)
        print(f'  saved best to {SCANNER_CKPT}')

print('Stage 1 training complete.')


In [ ]:
class RefinerDataset(Dataset):
    """Yields (snippet, proposal_norm, target_norm, class_idx) per GT segment.

    A 'noisy proposal' is simulated by jittering both endpoints of the GT to
    mimic Stage-1 outputs at inference time. The refiner learns to predict
    a residual that maps proposal_norm -> target_norm AND a class label."""

    def __init__(self, cache_path, window_size=64, jitter=12):
        self.cache       = torch.load(cache_path, weights_only=False)
        self.window_size = window_size
        self.jitter      = jitter
        self.snippets    = []   # (cache_idx, s, e, class_idx)

        for ci, item in enumerate(self.cache):
            # cache item: (fused, mask, vid, segments)
            for (s, e, cls) in item[3]:
                if s < e:
                    self.snippets.append((ci, s, e, cls))
        print(f'Refiner snippets: {len(self.snippets)}')

    def __len__(self): return len(self.snippets)

    def __getitem__(self, idx):
        ci, s, e, cls = self.snippets[idx]
        fused = self.cache[ci][0]
        T = fused.shape[0]
        W = self.window_size

        jit = self.jitter
        noisy_s = max(0, min(T-1, s + random.randint(-jit, jit)))
        noisy_e = max(0, min(T-1, e + random.randint(-jit, jit)))
        if noisy_e <= noisy_s: noisy_e = noisy_s + 1

        center  = (noisy_s + noisy_e) // 2
        w_start = max(0, center - W // 2)
        w_end   = w_start + W
        if w_end > T:
            w_end = T; w_start = max(0, w_end - W)

        snip = fused[w_start:w_end]
        if snip.shape[0] < W:
            pad = torch.zeros(W - snip.shape[0], snip.shape[1])
            snip = torch.cat([snip, pad], dim=0)

        proposal = torch.tensor([(noisy_s - w_start) / W, (noisy_e - w_start) / W],
                                dtype=torch.float32)
        target   = torch.tensor([(s       - w_start) / W, (e       - w_start) / W],
                                dtype=torch.float32)
        return snip, proposal, target, torch.tensor(cls, dtype=torch.long)


refiner_dataset = RefinerDataset(CACHE_TRAIN, window_size=WINDOW_SIZE, jitter=12)
refiner_loader  = DataLoader(refiner_dataset, batch_size=32, shuffle=True,
                             num_workers=0, pin_memory=False)
print(f'Snippets: {len(refiner_dataset)}, batches/epoch: {len(refiner_loader)}')

# Class balance check (helps spot rare classes)
from collections import Counter
class_counts = Counter(s[3] for s in refiner_dataset.snippets)
print(f'Class count range: min={min(class_counts.values())}  max={max(class_counts.values())}')


In [ ]:
class TransformerRefiner(nn.Module):
    """Outputs:
       - boundary residual (Δstart, Δend) in window-normalised units (raw, no sigmoid)
       - class logits over NUM_CLASSES
    Boundary head is initialised to zero so the model starts as identity over proposals.
    Class head is randomly initialised."""

    def __init__(self, num_classes, input_dim=2048, d_model=256, nhead=4,
                 num_layers=2, window_size=64, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.local_conv = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1)
        self.pos_embed  = nn.Parameter(torch.randn(1, window_size, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

        # Boundary head (zero-initialised — refiner starts as identity)
        self.bbox_head = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(),
            nn.Linear(64, 2),
        )
        nn.init.zeros_(self.bbox_head[-1].weight)
        nn.init.zeros_(self.bbox_head[-1].bias)

        # Classification head
        self.cls_head = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):                                  # x: (B, W, 2048)
        x = self.input_proj(x)
        x = self.local_conv(x.transpose(1, 2)).transpose(1, 2)
        x = x + self.pos_embed[:, :x.size(1)]
        x = self.encoder(x)
        h = x.mean(dim=1)                                  # (B, d_model)
        return self.bbox_head(h), self.cls_head(h)         # (B, 2), (B, NUM_CLASSES)


In [ ]:
class GIoULoss1D(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
    def forward(self, preds, targets):
        ps, pe = preds[:, 0], preds[:, 1]
        ts, te = targets[:, 0], targets[:, 1]
        s_lo = torch.minimum(ps, pe)
        e_hi = torch.maximum(ps, pe)
        inter = (torch.minimum(e_hi, te) - torch.maximum(s_lo, ts)).clamp(min=0)
        union = (e_hi - s_lo) + (te - ts) - inter + self.eps
        iou   = inter / union
        enc   = torch.maximum(e_hi, te) - torch.minimum(s_lo, ts) + self.eps
        giou  = iou - (enc - union) / enc
        return (1.0 - giou).mean()


In [ ]:
refiner = TransformerRefiner(num_classes=NUM_CLASSES, window_size=WINDOW_SIZE).to(DEVICE)
giou    = GIoULoss1D().to(DEVICE)
ce      = nn.CrossEntropyLoss()
opt2    = optim.AdamW(refiner.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler2 = torch.amp.GradScaler('cuda')

print(f'Refiner params: {sum(p.numel() for p in refiner.parameters())/1e6:.2f}M')

best_score = float('inf')

for epoch in range(EPOCHS_STAGE2):
    refiner.train()
    sums = {'total': 0.0, 'giou': 0.0, 'l1': 0.0, 'cls': 0.0, 'cls_acc': 0.0}
    n = 0
    pbar = tqdm(refiner_loader, desc=f'Stage2 {epoch+1}/{EPOCHS_STAGE2}',
                mininterval=1.0, maxinterval=10.0)

    for snip, proposal, target, cls_target in pbar:
        snip       = snip.to(DEVICE, non_blocking=True)
        proposal   = proposal.to(DEVICE, non_blocking=True)
        target     = target.to(DEVICE, non_blocking=True)
        cls_target = cls_target.to(DEVICE, non_blocking=True)

        opt2.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            delta, cls_logits = refiner(snip)
            refined           = proposal + delta

            l_giou = giou(refined.float(), target)
            l_l1   = F.smooth_l1_loss(delta.float(), target - proposal)
            l_cls  = ce(cls_logits.float(), cls_target)

            loss = W_GIOU * l_giou + W_L1 * l_l1 + W_CLS * l_cls

        scaler2.scale(loss).backward()
        scaler2.unscale_(opt2)
        torch.nn.utils.clip_grad_norm_(refiner.parameters(), GRAD_CLIP)
        scaler2.step(opt2); scaler2.update()

        with torch.no_grad():
            cls_acc = (cls_logits.argmax(-1) == cls_target).float().mean().item()
        sums['total']   += loss.item()
        sums['giou']    += l_giou.item()
        sums['l1']      += l_l1.item()
        sums['cls']     += l_cls.item()
        sums['cls_acc'] += cls_acc
        n += 1
        pbar.set_postfix(loss=f'{loss.item():.3f}', cls_acc=f'{cls_acc:.2f}')

    avgs = {k: v / max(1, n) for k, v in sums.items()}
    print(f'epoch {epoch+1}: total={avgs["total"]:.4f}  '
          f'giou={avgs["giou"]:.3f}  l1={avgs["l1"]:.3f}  '
          f'cls={avgs["cls"]:.3f}  cls_acc={avgs["cls_acc"]:.3f}')

    if avgs['total'] < best_score:
        best_score = avgs['total']
        torch.save(refiner.state_dict(), REFINER_CKPT)
        print(f'  saved best to {REFINER_CKPT}')

print('Stage 2 training complete.')


In [ ]:
def extract_proposals(probs, threshold=SCORE_THRESHOLD):
    mask = (probs > threshold).astype(np.int8)
    if mask.sum() == 0: return []
    padded = np.concatenate([[0], mask, [0]])
    diffs  = np.diff(padded.astype(np.int16))
    starts = np.where(diffs ==  1)[0].tolist()
    ends   = np.where(diffs == -1)[0].tolist()
    return list(zip(starts, ends))


def nms_1d_per_class(predictions, iou_thr=NMS_IOU_THR):
    """Apply NMS independently per class — standard for per-class mAP eval."""
    if not predictions: return []
    by_class = {}
    for p in predictions:
        by_class.setdefault(p['label'], []).append(p)

    kept = []
    for cls, preds in by_class.items():
        if not preds: continue
        segs   = np.asarray([p['segment'] for p in preds], dtype=np.float32)
        scores = np.asarray([p['score']   for p in preds], dtype=np.float32)
        order  = np.argsort(-scores)
        keep_idx = []
        while order.size > 0:
            i = order[0]; keep_idx.append(int(i))
            if order.size == 1: break
            rest = order[1:]
            s1, e1 = segs[i]
            s2, e2 = segs[rest, 0], segs[rest, 1]
            inter  = np.clip(np.minimum(e1, e2) - np.maximum(s1, s2), 0, None)
            union  = (e1 - s1) + (e2 - s2) - inter
            iou    = np.where(union > 0, inter / np.maximum(union, 1e-9), 0.0)
            order  = rest[iou < iou_thr]
        kept.extend([preds[k] for k in keep_idx])
    return kept


In [ ]:
mamba_model = BiMambaGlobalScanner().to(DEVICE)
mamba_model.load_state_dict(torch.load(SCANNER_CKPT, map_location=DEVICE, weights_only=True))
mamba_model.eval()

refiner = TransformerRefiner(num_classes=NUM_CLASSES, window_size=WINDOW_SIZE).to(DEVICE)
refiner.load_state_dict(torch.load(REFINER_CKPT, map_location=DEVICE, weights_only=True))
refiner.eval()

test_cache  = torch.load(CACHE_TEST, weights_only=False)
predictions = {}

with torch.no_grad():
    for fused, _, vid, _ in tqdm(test_cache, desc='Inference', mininterval=1.0):
        T = fused.shape[0]
        feats = fused.unsqueeze(0).to(DEVICE)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = mamba_model(feats)
        probs = torch.sigmoid(logits.float()).squeeze(0).cpu().numpy()

        proposals = extract_proposals(probs, SCORE_THRESHOLD)
        if not proposals:
            predictions[vid] = []; continue

        per_video = []
        for ps, pe in proposals:
            center  = (ps + pe) // 2
            w_start = max(0, center - WINDOW_SIZE // 2)
            w_end   = w_start + WINDOW_SIZE
            if w_end > T:
                w_end = T; w_start = max(0, w_end - WINDOW_SIZE)
            snip = fused[w_start:w_end]
            if snip.shape[0] < WINDOW_SIZE:
                pad = torch.zeros(WINDOW_SIZE - snip.shape[0], snip.shape[1])
                snip = torch.cat([snip, pad], dim=0)

            with torch.amp.autocast('cuda', dtype=torch.float16):
                delta, cls_logits = refiner(snip.unsqueeze(0).to(DEVICE))
            delta      = delta.float().squeeze(0).cpu().numpy()
            cls_probs  = torch.softmax(cls_logits.float(), dim=-1).squeeze(0).cpu().numpy()
            cls_idx    = int(cls_probs.argmax())
            cls_score  = float(cls_probs[cls_idx])

            # Apply boundary residual
            prop_s_norm = (ps - w_start) / WINDOW_SIZE
            prop_e_norm = (pe - w_start) / WINDOW_SIZE
            r_s_norm = prop_s_norm + delta[0]
            r_e_norm = prop_e_norm + delta[1]
            r_s_norm, r_e_norm = min(r_s_norm, r_e_norm), max(r_s_norm, r_e_norm)
            r_s = w_start + r_s_norm * WINDOW_SIZE
            r_e = w_start + r_e_norm * WINDOW_SIZE

            seg_start = float(r_s * CLIP_LENGTH / FPS)
            seg_end   = float(r_e * CLIP_LENGTH / FPS)
            if seg_end <= seg_start: continue

            # Final score combines actionness (Stage 1) and classification confidence
            actionness = float(probs[ps:pe].mean())
            score      = actionness * cls_score
            per_video.append({
                'segment': [seg_start, seg_end],
                'score':   score,
                'label':   CLASS_NAMES[cls_idx],
            })

        predictions[vid] = nms_1d_per_class(per_video)

with open(PRED_FILE, 'w') as f:
    json.dump(predictions, f)
print(f'Saved {sum(len(v) for v in predictions.values())} segments across '
      f'{len(predictions)} videos -> {PRED_FILE}')


In [ ]:
# Per-class mAP @ {0.3, 0.4, 0.5, 0.6, 0.7} matching the official THUMOS14 evaluator.
# For each (class, IoU threshold):
#   1. Sort that class's predictions by score globally (across all test videos).
#   2. For each prediction in order: TP if it matches an unmatched GT of same class
#      and same video at IoU >= threshold; else FP.
#   3. Compute precision/recall as cumulative; AP is the area under the PR curve.
# mAP = mean over classes; reported number is mean over IoU thresholds too.

IOU_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]


def iou_1d(a, b):
    inter = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    union = (a[1] - a[0]) + (b[1] - b[0]) - inter
    return inter / union if union > 0 else 0.0


def voc_ap(rec, prec):
    """VOC07-style AP — interpolated 11-point. Standard for THUMOS14 historically."""
    ap = 0.0
    for t in np.arange(0.0, 1.01, 0.1):
        p = prec[rec >= t].max() if (rec >= t).any() else 0.0
        ap += p / 11.0
    return ap


def compute_map(predictions, gt_data, test_vids, classes, iou_thr):
    aps = {}
    for cls in classes:
        # Gather per-video GT for this class
        gt_by_vid = {}
        n_gt = 0
        for vid in test_vids:
            entry = gt_data.get('database', {}).get(vid, {})
            segs  = []
            for ann in entry.get('annotations', []):
                if _label_of(ann) == cls:
                    segs.append([float(ann['segment'][0]), float(ann['segment'][1])])
            gt_by_vid[vid] = {'segs': segs, 'matched': [False] * len(segs)}
            n_gt += len(segs)
        if n_gt == 0:
            aps[cls] = float('nan'); continue

        # Gather all predictions for this class
        all_preds = []
        for vid in test_vids:
            for p in predictions.get(vid, []):
                if p.get('label') == cls:
                    all_preds.append((vid, p['segment'], p['score']))
        if not all_preds:
            aps[cls] = 0.0; continue

        all_preds.sort(key=lambda x: -x[2])
        tp = np.zeros(len(all_preds))
        fp = np.zeros(len(all_preds))
        for i, (vid, seg, _) in enumerate(all_preds):
            entry = gt_by_vid[vid]
            best_j, best_iou = -1, 0.0
            for j, g in enumerate(entry['segs']):
                if entry['matched'][j]: continue
                v = iou_1d(seg, g)
                if v > best_iou: best_j, best_iou = j, v
            if best_iou >= iou_thr and best_j >= 0:
                entry['matched'][best_j] = True
                tp[i] = 1
            else:
                fp[i] = 1

        cum_tp = np.cumsum(tp); cum_fp = np.cumsum(fp)
        rec  = cum_tp / max(1, n_gt)
        prec = cum_tp / np.maximum(cum_tp + cum_fp, 1e-9)
        aps[cls] = voc_ap(rec, prec)
    return aps


with open(f'{DATA_ROOT}/gt.json') as f:    gt_data = json.load(f)
with open(PRED_FILE) as f:                 preds   = json.load(f)
with open(f'{DATA_ROOT}/split_test.txt') as f:
    test_vids = set(l.strip() for l in f if l.strip())

# Filter classes that appear in test split (otherwise mAP is undefined)
classes_in_test = set()
for vid in test_vids:
    for ann in gt_data.get('database', {}).get(vid, {}).get('annotations', []):
        classes_in_test.add(_label_of(ann))
eval_classes = sorted([c for c in CLASS_NAMES if c in classes_in_test])
print(f'Evaluating on {len(eval_classes)} classes present in test split')
print()

results_by_thr = {}
for thr in IOU_THRESHOLDS:
    aps = compute_map(preds, gt_data, test_vids, eval_classes, thr)
    valid_aps = [v for v in aps.values() if not math.isnan(v)]
    mAP = float(np.mean(valid_aps)) if valid_aps else 0.0
    results_by_thr[thr] = (aps, mAP)
    print(f'  mAP @ IoU={thr:.1f}: {mAP:.4f}')

mean_mAP = float(np.mean([r[1] for r in results_by_thr.values()]))
print()
print(f'Average mAP across IoU thresholds: {mean_mAP:.4f}')

# Per-class breakdown at IoU 0.5 (the conventional reporting threshold)
print()
print('Per-class AP @ IoU 0.5:')
aps_at_half = results_by_thr[0.5][0]
for c in sorted(aps_at_half.keys(), key=lambda x: -aps_at_half[x]):
    if not math.isnan(aps_at_half[c]):
        print(f'  {c:30s} {aps_at_half[c]:.3f}')


In [ ]:
# Class-agnostic F1 — same as v2's eval. Use to compare apples-to-apples with
# your previous Stage-1-only run (F1@0.5 ≈ 0.49 baseline). Note this metric
# treats predictions as correct as long as IoU matches some GT segment of any
# class — far easier than per-class mAP, so numbers will look higher.

with open(f'{DATA_ROOT}/gt.json') as f:    gt_data = json.load(f)
with open(PRED_FILE) as f:                 preds   = json.load(f)
with open(f'{DATA_ROOT}/split_test.txt') as f:
    test_vids = set(l.strip() for l in f if l.strip())

print(f'{"thr":>5} {"TP":>6} {"FP":>6} {"GT":>6} {"P":>7} {"R":>7} {"F1":>7}')
print('-' * 50)
for thr in IOU_THRESHOLDS:
    tp = fp = total_gt = 0
    for vid, gt_entry in gt_data.get('database', {}).items():
        if vid not in test_vids: continue
        gt_segs = [[float(s) for s in a['segment']] for a in gt_entry.get('annotations', [])]
        total_gt += len(gt_segs)
        matched = [False] * len(gt_segs)
        for p in sorted(preds.get(vid, []), key=lambda x: -x['score']):
            best_i, best_iou = -1, 0.0
            for i, g in enumerate(gt_segs):
                if matched[i]: continue
                v = iou_1d(p['segment'], g)
                if v > best_iou: best_i, best_iou = i, v
            if best_iou >= thr and best_i >= 0:
                matched[best_i] = True; tp += 1
            else:
                fp += 1
    P = tp / max(1, tp + fp); R = tp / max(1, total_gt)
    F1 = 2*P*R / max(1e-9, P + R)
    print(f'{thr:>5} {tp:>6} {fp:>6} {total_gt:>6} {P:>7.3f} {R:>7.3f} {F1:>7.3f}')
